# Concept Grouping Walkthrough

This notebook shows the intended v1 user workflow for grouped interpretation:

1. compute attributions
2. call `group_attributions(...)` once
3. inspect ranked concept rows

It keeps only the representative cases:

- diagnosis grouping into CCS concepts
- standardized drug-code grouping into ATC classes
- same-vocabulary ancestor grouping
- a real PyHealth batch workflow with StageNet and Integrated Gradients
    


## Imports and tiny helpers

The notebook uses small one-sample batch wrappers for the toy cases so every section uses the same public API.
    


In [1]:
from __future__ import annotations

import pandas as pd
import torch
from IPython.display import display

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.interpret import group_attributions
from pyhealth.interpret.methods import IntegratedGradients
from pyhealth.models import StageNet


def show_rows(rows):
    display(pd.DataFrame(rows))


def build_single_feature_batch(feature_key, tokens, scores):
    return {feature_key: [tokens]}, {feature_key: torch.tensor([scores])}


def build_sample_dataset():
    samples = [
        {
            'patient_id': 'patient-0',
            'visit_id': 'visit-0',
            'conditions': ([0.0, 1.0], [['428.0', '428.1'], ['250.00']]),
            'label': 1,
        },
        {
            'patient_id': 'patient-1',
            'visit_id': 'visit-1',
            'conditions': ([0.0, 2.0], [['401.9'], ['428.0', '250.00']]),
            'label': 0,
        },
        {
            'patient_id': 'patient-2',
            'visit_id': 'visit-2',
            'conditions': ([0.0], [['428.0', '401.9']]),
            'label': 1,
        },
    ]
    return create_sample_dataset(
        samples=samples,
        input_schema={'conditions': 'stagenet'},
        output_schema={'label': 'binary'},
        dataset_name='concept_grouping_walkthrough',
    )
    


## 1. Diagnosis grouping: `ICD9CM -> CCSCM`

This is the most common interpretation case for EHR prediction tasks. The output is already ranked and readable, so no second formatting step is needed.
    


In [2]:
dx_batch, dx_attr = build_single_feature_batch(
    'conditions',
    ['428.0', '428.1', '250.00', '401.9'],
    [0.12, -0.08, 0.21, 0.05],
)

dx_rows = group_attributions(
    batch=dx_batch,
    attributions=dx_attr,
    feature_key='conditions',
    source_vocab='ICD9CM',
    group_by='CCSCM',
    topk=5,
)
show_rows(dx_rows)
    


,rank,group_id,label,score,tokens,token_labels
0,1,49,Diabetes mellitus without complication,0.21,[250.00],[Diabetes mellitus without mention of complica...
1,2,108,Congestive heart failure; nonhypertensive,0.20,"[428.0, 428.1]","[Congestive heart failure, unspecified, Left h..."
2,3,98,Essential hypertension,0.05,[401.9],[Unspecified essential hypertension]


## 2. Standardized drug-code grouping: `NDC -> ATC`

This is the representative medication example. `target_kwargs` stays public because ATC level selection is a real workflow need.
    


In [3]:
drug_batch, drug_attr = build_single_feature_batch(
    'drugs',
    ['00527051210', '00536338101', '63323026201'],
    [0.35, 0.40, 0.20],
)

drug_rows = group_attributions(
    batch=drug_batch,
    attributions=drug_attr,
    feature_key='drugs',
    source_vocab='NDC',
    group_by='ATC',
    topk=5,
    target_kwargs={'level': 3},
)
show_rows(drug_rows)
    


,rank,group_id,label,score,tokens,token_labels
0,1,A06A,DRUGS FOR CONSTIPATION,0.40,[00536338101],[bisacodyl 5 MG Delayed Release Oral Tablet]
1,2,A11C,"VITAMIN A AND D, INCL. COMBINATIONS OF THE TWO",0.35,[00527051210],[Ergocalciferol 50000 UNT Oral Capsule]
2,3,B01A,ANTITHROMBOTIC AGENTS,0.20,[63323026201],"[heparin sodium, porcine 5000 UNT/ML Injectabl..."


## 3. Ancestor grouping inside one vocabulary

Not every interpretation path needs a cross-vocabulary map. Sometimes one step up in the source ontology is enough.
    


In [4]:
ancestor_batch, ancestor_attr = build_single_feature_batch(
    'conditions',
    ['428.0', '428.1', '250.00'],
    [0.30, 0.40, 0.20],
)

ancestor_rows = group_attributions(
    batch=ancestor_batch,
    attributions=ancestor_attr,
    feature_key='conditions',
    source_vocab='ICD9CM',
    group_by='ancestor',
    topk=5,
    ancestor_level=1,
)
show_rows(ancestor_rows)
    


,rank,group_id,label,score,tokens,token_labels
0,1,428,Heart failure,0.7,"[428.0, 428.1]","[Congestive heart failure, unspecified, Left h..."
1,2,250.0,Diabetes mellitus without mention of complication,0.2,[250.00],[Diabetes mellitus without mention of complica...


## 4. PyHealth-native workflow: `batch + attributions -> grouped rows`

This is the intended real usage pattern: get a dataloader batch, compute attributions, then call the same public API once.
    


In [5]:
torch.manual_seed(7)

dataset = build_sample_dataset()
model = StageNet(dataset=dataset, embedding_dim=16, chunk_size=8, levels=2)
model.eval()

batch = next(iter(get_dataloader(dataset, batch_size=1, shuffle=False)))
ig = IntegratedGradients(model, use_embeddings=True)
attributions = ig.attribute(**batch, steps=4, target_class_idx=1)

summary = group_attributions(
    batch=batch,
    attributions=attributions,
    dataset=dataset,
    feature_key='conditions',
    source_vocab='ICD9CM',
    group_by='CCSCM',
    topk=5,
)
show_rows(summary)
    


Label label vocab: {0: 0, 1: 1}


,rank,group_id,label,score,tokens,token_labels
0,1,108,Congestive heart failure; nonhypertensive,0.047680,"[428.0, 428.1]","[Congestive heart failure, unspecified, Left h..."
1,2,49,Diabetes mellitus without complication,0.044383,[250.00],[Diabetes mellitus without mention of complica...
